In [4]:
from dotenv import load_dotenv
import os
load_dotenv()  # .env 파일 로드
api_key = os.getenv("IBMQ_API_KEY")  # from .env file

from qiskit_ibm_runtime import QiskitRuntimeService
 
service = QiskitRuntimeService(channel="ibm_quantum", token=api_key)

In [5]:
from qiskit_ibm_runtime import QiskitRuntimeService
 
# Save an IBM Quantum account and set it as your default account.
QiskitRuntimeService.save_account(channel="ibm_quantum", token=api_key, set_as_default=True, overwrite=True)
 
# Load saved credentials
service = QiskitRuntimeService()

In [6]:
service.backends()

[<IBMBackend('ibm_brisbane')>,
 <IBMBackend('ibm_kyiv')>,
 <IBMBackend('ibm_sherbrooke')>]

In [8]:
 from qiskit import QuantumCircuit
 from qiskit_ibm_runtime import QiskitRuntimeService, Sampler
 
 # Create empty circuit
 example_circuit = QuantumCircuit(2)
 example_circuit.measure_all()
 
 # You'll need to specify the credentials when initializing QiskitRuntimeService, if they were not previously saved.
 service = QiskitRuntimeService()
 backend = service.backend("ibm_sherbrooke")
 job = Sampler(backend).run([example_circuit])
 print(f"job id: {job.job_id()}")
 result = job.result()
 print(result)

job id: cz9zxs7tp60g008h10wg
PrimitiveResult([SamplerPubResult(data=DataBin(meas=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}})], metadata={'execution': {'execution_spans': ExecutionSpans([{'__type__': 'DoubleSliceSpan', '__value__': {'start': datetime.datetime(2025, 3, 14, 10, 3, 20, 126977), 'stop': datetime.datetime(2025, 3, 14, 10, 3, 23, 213799), 'data_slices': {'0': [[4096], 0, 1, 0, 4096]}}}])}, 'version': 2})


In [10]:
from qiskit import QuantumCircuit, transpile
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.visualization import plot_histogram
import torch
import numpy as np
from modules.utils import read_args
import os 


epoch = 300
trial_name = "True_np5_nl10_biased_diamond_True_Mar08_18_52_11"
base_dir = os.path.join('./2d_runs', trial_name)
args_file_path = os.path.join(base_dir, 'args.txt')
param_file_path = os.path.join(base_dir, 'params', f'generator_params_epoch{epoch}.pth')
generator_params = torch.load(param_file_path, weights_only=True).detach().numpy()

n_qubits, code_qubits, n_layers, SEED = read_args(args_file_path, "n_qubits", "code_qubits", "n_layers", "seed")
print(f"n_qubits: {n_qubits}, n_layers: {n_layers}, SEED: {SEED}")


# Generate a 5-qubit simulated backend
backend = GenericBackendV2(num_qubits=n_qubits)

def QGAN2_ibmq(circuit, inputs, params):
    assert(len(inputs) == params.shape[1])
    n_qubits = len(inputs)
    n_layers = params.shape[0]

    for i in range(n_qubits):
        circuit.ry(inputs[i] * np.pi/2, i)
    for l in range(n_layers):
        for i in range(n_qubits):
            circuit.ry(params[l, i, 0], i)
        if l < n_layers-1:
            for i in range(n_qubits-1):
                circuit.cx(i, i+1)
            circuit.cx(n_qubits-1, 0)

# Create a simple circuit
circuit = QuantumCircuit(n_qubits)
z = np.random.uniform(-SEED, SEED, (n_qubits))
QGAN2_ibmq(circuit, z, generator_params)
circuit.measure_all()
# Transpile the ideal circuit to a circuit that can be directly executed by the backend
transpiled_circuit = transpile(circuit, backend)
# Run the transpiled circuit using the simulated backend
job = backend.run(transpiled_circuit)
counts = job.result().get_counts()
arr = np.zeros(2**n_qubits, dtype=int)
for bitstr, cnt in counts.items():
    arr[int(bitstr, 2)] = cnt
plot_histogram(counts)
print(arr, arr.sum())

n_qubits: 5, n_layers: 10, SEED: 0.5
[139  14  11   4  36  19  81  24   5  11  23  39  11  11  26   9  13   2
  24  12   7   7  55 214  23  14  52  12   7  41  58  20] 1024


In [20]:
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches


epoch = 300
trial_name = "True_np5_nl10_biased_diamond_True_Mar08_18_52_16"
base_dir = os.path.join('./2d_runs', trial_name)
args_file_path = os.path.join(base_dir, 'args.txt')
param_file_path = os.path.join(base_dir, 'params', f'generator_params_epoch{epoch}.pth')
generator_params = torch.load(param_file_path, weights_only=True).detach().numpy()
n_qubits, code_qubits, n_layers, SEED = read_args(args_file_path, "n_qubits", "code_qubits", "n_layers", "seed")
print(f"n_qubits: {n_qubits}, n_layers: {n_layers}, SEED: {SEED}")

output_records = []
input_records = []

rep = 1000

backend = GenericBackendV2(num_qubits=n_qubits)

postprocessing_dir = os.path.join(base_dir, 'postprocessing')
os.makedirs(postprocessing_dir, exist_ok=True)
output_img_dir = os.path.join(base_dir, 'postprocessing', 'ibmq_simulator')
os.makedirs(output_img_dir, exist_ok=True)

def bitwise_sums(arr):
    n = len(arr).bit_length() - 1  # 비트 길이를 계산하여 반복 횟수를 정함
    sums = torch.zeros(n)  # 결과를 저장할 텐서
    for bit in range(n):
        # 조건에 맞는 인덱스 선택을 위해 i-th 비트를 검사
        mask = (torch.arange(len(arr), device=arr.device) >> bit) & 1
        sums[bit] = arr[mask.bool()].sum()  # 조건에 맞는 원소들의 합산
    return sums

for i in tqdm(range(rep)):
    circuit = QuantumCircuit(n_qubits)
    z = np.random.uniform(-SEED, SEED, (n_qubits)) # input
    QGAN2_ibmq(circuit, z, generator_params)
    circuit.measure_all()
    
    transpiled_circuit = transpile(circuit, backend)
    # Run the transpiled circuit using the simulated backend
    job = backend.run(transpiled_circuit)
    counts = job.result().get_counts()
    arr = np.zeros(2**n_qubits, dtype=int)
    for bitstr, cnt in counts.items():
        arr[int(bitstr, 2)] = cnt
    probabilities = arr / arr.sum()
    probabilities = bitwise_sums(probabilities)
    #print("i = ", i, probabilities )
    
    output_records.append(probabilities)
    input_records.append(z[-code_qubits:])
    if i % 10 != 9:
        continue

    outputs = np.array(output_records)
    inputs = np.array(input_records)

    # 시각화
    for code_ind in range(code_qubits):
        plt.figure(figsize=(12,10))
        plt.scatter(outputs[:, 0], outputs[:, 1], c=inputs[:, code_ind], cmap='RdYlBu', alpha=0.4, s=10)
        plt.colorbar()  # 색상 막대 추가
        plt.title(f'code{code_ind} (size={i+1}, ibmq simulator)')
        plt.xlim((0, 1))
        plt.ylim((0, 1))
        ax = plt.gca()
        
        # 중심 (0.6, 0.6), 팔 길이 0.2sqrt(2)인 다이아몬드 추가
        arm = 0.2 * np.sqrt(2)
        circle = patches.Polygon([[0.6+arm, 0.6], [0.6, 0.6-arm], [0.6-arm, 0.6], [0.6, 0.6+arm]], closed=True, fill=False, edgecolor='red')
        ax.add_patch(circle)

        save_dir = os.path.join(output_img_dir, f'ibmq_simulator_code{code_ind}_{i+1}.png')
        plt.savefig(save_dir)
        plt.close()

# 결과 가져오기
print("IonQ noisy simulator 결과:")
print(job.histogram(key='x'))

n_qubits: 5, n_layers: 10, SEED: 0.5


  0%|          | 0/1000 [00:00<?, ?it/s]

100%|██████████| 1000/1000 [06:41<00:00,  2.49it/s]

IonQ noisy simulator 결과:


AttributeError: 'AerJob' object has no attribute 'histogram'

In [5]:
counts

{'001': 520, '000': 498, '101': 3, '010': 1, '011': 2}